# Peajes colombianos: análisis histórico (2025)
Versión editorial del notebook más completo. Las salidas se retiraron para reducir peso.
**No es una evaluación corregida**: la cascada RF ajusta la primera etapa con todos los datos y las ANN preprocesan antes de separar. Consulte el informe.
Las métricas históricas respaldadas están en `resultados/metricas_historicas_trafico.csv`.
Cambios editoriales: lectura local, API `sparse_output`, mapas redundantes retirados, coordenadas corregidas en el mapa conservado y mensaje de métricas del ensayo de learning rate corregido.


# **Notebook poster Workshop en Estadística**

## 🧾 Data Dictionary — Dataset de Peajes en Colombia

| **Columna** | **Tipo de dato** | **Descripción** | **Ejemplo** |
|--------------|------------------|------------------|--------------|
| **IdPeaje** | `int64` | Identificador único del peaje dentro del sistema nacional de concesiones viales. | `1` |
| **Peaje** | `object` | Nombre del peaje. Puede repetirse para distintos periodos o categorías tarifarias. | `ALVARADO` |
| **IdCategoriaTarifa** | `object` | Categoría del vehículo según su tipo o número de ejes. Generalmente sigue una clasificación I, II, III, IV, V, etc. | `II` |
| **FechaDesde** | `object` (→ conviene convertir a `datetime`) | Fecha inicial de vigencia de la tarifa y registro de tráfico. | `2015-08-21` |
| **FechaHasta** | `object` (→ conviene convertir a `datetime`) | Fecha final de vigencia de la tarifa y registro de tráfico. | `2015-08-31` |
| **ValorTarifa** | `int64` | Valor de la tarifa cobrada en pesos colombianos (COP) para la categoría correspondiente. | `7600` |
| **Trafico** | `int64` | Número total de vehículos registrados en el periodo y categoría indicados. | `9930` |
| **TraficoEvasores** | `float64` | Número estimado de vehículos que evadieron el peaje (no pagaron la tarifa). Puede contener valores nulos. | `0.0` |
| **TraficoExentos787** | `float64` | Número de vehículos exentos según la Ley 787 de 2002 (vehículos de servicio público, oficiales, de emergencia, etc.). Puede contener valores nulos. | `70.0` |
| **Lat** | `float64` | Latitud geográfica del peaje. | `4.507609` |
| **Lon** | `float64` | Longitud geográfica del peaje. | `-74.991661` |

### 📌 Notas
- Los campos **FechaDesde** y **FechaHasta** permiten definir intervalos de tiempo, por lo que se pueden agrupar datos por mes o año.  
- Las columnas **TraficoEvasores** y **TraficoExentos787** pueden contener valores nulos; conviene imputarlos con 0 o analizarlos aparte.  
- Los pares (**Lat**, **Lon**) permiten generar mapas o visualizaciones geográficas.  
- Variables derivadas sugeridas:  
  - `DuracionPeriodo = FechaHasta - FechaDesde`  
  - `TraficoPromedioDiario = Trafico / DuracionPeriodo`


## 👨🏽‍💻🐍**Work:**

### 0️⃣ Primeros pasos

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
from pathlib import Path
RAIZ = Path.cwd() if (Path.cwd() / 'datos').exists() else Path.cwd().parent
df = pd.read_csv(RAIZ / 'datos' / 'peajes_2019_2022.csv')
df.drop(columns=['Unnamed: 0'], errors='ignore', inplace=True)
df.info()


In [ ]:
df

In [ ]:
## --- Conversión de tipos de datos ---

# Fechas
df["FechaDesde"] = pd.to_datetime(df["FechaDesde"], format="%Y-%m-%d", errors="coerce")
df["FechaHasta"] = pd.to_datetime(df["FechaHasta"], format="%Y-%m-%d", errors="coerce")

# Categóricas
df["Peaje"] = df["Peaje"].astype("category")
df["IdCategoriaTarifa"] = df["IdCategoriaTarifa"].astype("category")
df["TipoVehiculo"] = df["TipoVehiculo"].astype("category")




# --- Final check ---
print("**Resumen del DataFrame:**\n")
print(df.info())
print("\n\n**Primeras filas:**")
df.head()

In [ ]:
df_filtered_2019_cats_oficiales = df.copy()

### 1️⃣ EDA:

In [ ]:
import plotly.express as px

fig = px.bar(
    df_filtered_2019_cats_oficiales.groupby("TipoVehiculo", observed=True)["Trafico"].sum().reset_index(),
    x="Trafico",
    y="TipoVehiculo",
    orientation="h",
    title="Distribución del tráfico total por tipo de vehículo",
    text_auto=True,
    color="TipoVehiculo",
)
fig.update_layout(showlegend=False)
fig.show()

In [ ]:
fig = px.bar(
    df_filtered_2019_cats_oficiales.groupby("TipoVehiculo", observed=True)["IngresoTotal"].sum().reset_index(),
    x="IngresoTotal",
    y="TipoVehiculo",
    orientation="h",
    title="Ingresos totales por tipo de vehículo",
    text_auto=True,
    color="TipoVehiculo",
)
fig.update_layout(showlegend=False)
fig.show()

In [ ]:
plt.figure(figsize=(10,6))
sns.scatterplot(
    data=df_filtered_2019_cats_oficiales,
    x="ValorTarifa",
    y="TraficoPromedioDiario",
    hue="TipoVehiculo",
    alpha=0.7
)
plt.title("Relación entre Valor del Peaje y Tráfico Promedio Diario")
plt.xlabel("Valor Tarifa (COP)")
plt.ylabel("Tráfico Promedio Diario")
plt.legend(title="Tipo de Vehículo", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.grid(True)
# guardar la imagen:
plt.savefig(RAIZ / "resultados" / "relacion_valor_tarifa_&_trafico_promedio_diario.png", dpi=200, bbox_inches='tight')
plt.show() # mostrarla

In [ ]:
top_peajes = (
    df_filtered_2019_cats_oficiales.groupby("Peaje", observed=True)["TraficoPromedioDiario"]
    .mean()
    .nlargest(10)
    .reset_index()
)

fig = px.bar(
    top_peajes,
    x="TraficoPromedioDiario",
    y="Peaje",
    orientation="h",
    title="Top 10 Peajes con mayor tráfico promedio diario (2019)",
    text_auto=True,
    color="TraficoPromedioDiario",
)
fig.update_layout(showlegend=False)
fig.show()

In [ ]:
import matplotlib.patches as patches

# Seleccionar columnas numéricas relevantes
numeric_cols = set(df_filtered_2019_cats_oficiales.select_dtypes(include=['float64', 'int64']).columns)
cols_a_remover = {'IdPeaje', 'Lat', 'Lon', 'DuracionPeriodo(DIAS)'}
numeric_cols = list(numeric_cols - cols_a_remover)
del cols_a_remover

# Calcular la matriz de correlación
corr_matrix = df_filtered_2019_cats_oficiales[numeric_cols].corr()

# Crear máscara para ocultar el triángulo superior (sin incluir diagonal)
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)

# Configurar figura
plt.figure(figsize=(9, 7))
ax = plt.gca()

# Dibujar el heatmap (solo triángulo inferior)
sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True,
    cmap='coolwarm',
    fmt=".4f",
    center=0,
    square=True,
    linewidths=0.5,
    cbar_kws={"shrink": .8},
    ax=ax
)

# Colorear la diagonal con un tono suave para distinguirla
cmap_diag = plt.cm.Greys
diag_color = cmap_diag(0.85)

for i in range(len(corr_matrix)):
    square = patches.Rectangle(
        (i, i), 1, 1,
        linewidth=0,
        edgecolor='none',
        facecolor=diag_color,
        alpha=0.4
    )
    ax.add_patch(square)

# Rotar etiquetas y mejorar legibilidad
plt.xticks(rotation=35, ha='right')
# plt.yticks(rotation=45)
plt.title("Matriz de correlación — Dataset de Peajes", pad=20)
plt.tight_layout()
# guardar la imagen:
plt.savefig(RAIZ / "resultados" / "matriz_de_correlacion_peajes.png", dpi=200, bbox_inches='tight')
plt.show()  # mostrarla

In [ ]:
fig = px.scatter_map(
    df_filtered_2019_cats_oficiales
        .groupby(["Peaje", "Lat", "Lon"], as_index=False, observed=True)["Trafico"]
        .sum(),

    lat="Lon",
    lon="Lat",
    color="Trafico",
    hover_name="Peaje",
    title="Mapa de tráfico total por peaje",
    color_continuous_scale="plasma",
    zoom=4.5,
    opacity=0.8,
    map_style="carto-voyager"
)

fig.show()

#### Visualize traffic over time



In [ ]:
# Generar la serie temporal mensual (suma total por mes)
df_monthly_traffic = df_filtered_2019_cats_oficiales.dropna(subset=['FechaDesde']).resample('ME', on='FechaDesde')['Trafico'].sum()

# Crear figura con tamaño adecuado
plt.figure(figsize=(14, 7))

# Graficar en millones para mejor lectura
(df_monthly_traffic / 1_000_000).plot(
    linewidth=2.5,
    color='steelblue',
    marker='o',
    markersize=4,
    label='Tráfico Total Mensual'
)

# Etiquetas
plt.xlabel('Fecha', fontsize=12, c='#FF8904')
plt.ylabel('Tráfico Total (millones de vehículos/mes)', fontsize=12, c='#FF8904')
plt.title('Evolución del Tráfico Mensual Nacional (2019 – 2022)', fontsize=14, fontweight='bold', color='darkred')

# Añadir cuadrícula
plt.grid(True, which='both', linestyle='--', alpha=0.6)

# Rotar fechas para que se lean bien
#plt.xticks(rotation=45, ha='right')

# Mejorar diseño
plt.tight_layout()

# Opcional: guardar con alta calidad
plt.savefig(RAIZ / "resultados" / "serie_tiempo_trafico_nacional.png", dpi=200, bbox_inches='tight')

# Mostrar gráfico
plt.show()

#### Analyze revenue by vehicle category


In [ ]:
revenue_by_category = df_filtered_2019_cats_oficiales.groupby('TipoVehiculo', observed=True)['IngresoTotal'].sum().sort_values(ascending=False)

plt.figure(figsize=(10, 6))
revenue_by_category.plot(kind='bar')
plt.xlabel('Vehicle Type')
plt.ylabel('Total Revenue')
plt.title('Total Revenue by Vehicle Type (Official Categories)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

#### Explore traffic distribution by toll booth


In [ ]:
toll_booth_traffic = df_filtered_2019_cats_oficiales.groupby('Peaje', observed=True)['Trafico'].sum().sort_values(ascending=False)
top_n_toll_booths = toll_booth_traffic.head(15)

plt.figure(figsize=(12, 6))
top_n_toll_booths.plot(kind='bar')
plt.xlabel('Toll Booth')
plt.ylabel('Total Traffic')
plt.title('Total Traffic for Top 15 Toll Booths (Official Categories, 2019 Onwards)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
toll_booth_traffic = df_filtered_2019_cats_oficiales.groupby('Peaje', observed=True)['Trafico'].sum().sort_values(ascending=False)
top_n_toll_booths = toll_booth_traffic.head(10)

plt.figure(figsize=(12, 6))
top_n_toll_booths.plot(kind='bar')
plt.xlabel('Toll Booth')
plt.ylabel('Total Traffic')
plt.title('Top 10 de peajes con más tráfico total')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
# guardar la imagen:
plt.savefig(RAIZ / "resultados" / "top_10_peajes_por_trafico_total.png", dpi=200, bbox_inches='tight')
plt.show()  # mostrarla

#### Investigate tariffs and traffic



In [ ]:
plt.figure(figsize=(12, 8))
sns.scatterplot(data=df_filtered_2019_cats_oficiales, x='ValorTarifa', y='TraficoPromedioDiario', hue='IdCategoriaTarifa', alpha=0.6)
plt.xlabel('Valor Tarifa (COP)')
plt.ylabel('Trafico Promedio Diario (Vehículos/Día)')
plt.title('Relación entre Valor Tarifa y Tráfico Promedio Diario por Categoría de Vehículo (2019 en adelante)')
plt.legend(title='Categoría de Vehículo')
plt.grid(True)
plt.show()

### 2️⃣ Modelos de Aprendizaje Supervisado (para regresión)

#### Predecir `TraficoPromedioDiario`:

In [ ]:
df_filtered_2019_cats_oficiales.head()

In [ ]:
df_filtered_2019_cats_oficiales.info()

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

In [ ]:
features = ['ValorTarifa', 'TipoVehiculo', 'Peaje', 'DuracionPeriodo(DIAS)', ]
target = 'TraficoPromedioDiario'

In [ ]:
#target = 'TraficoPromedioDiario'
print(df_filtered_2019_cats_oficiales['TraficoPromedioDiario'].nlargest(1))
print(df_filtered_2019_cats_oficiales['TraficoPromedioDiario'].nsmallest(1))

In [ ]:
X = df_filtered_2019_cats_oficiales[features]
y = df_filtered_2019_cats_oficiales[target]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
numeric_features = ['ValorTarifa']
categorical_features = ['TipoVehiculo', 'Peaje', 'DuracionPeriodo(DIAS)']

preprocessor = ColumnTransformer(
transformers=[
('num', StandardScaler(), numeric_features),
('cat', OneHotEncoder(drop='first', sparse_output=False), categorical_features)
])

In [ ]:
models = {
'Linear Regression': Pipeline([
('preprocessor', preprocessor),
('regressor', LinearRegression())
]),
'Random Forest': Pipeline([
('preprocessor', preprocessor),
('regressor', RandomForestRegressor(n_estimators=100, random_state=42))
]),
'XGBoost': Pipeline([
('preprocessor', preprocessor),
('regressor', xgb.XGBRegressor(random_state=42))
]),
'SVR': Pipeline([
('preprocessor', preprocessor),
('regressor', SVR(kernel='rbf'))
])
}

In [ ]:
results = {}

for name, model in models.items():
    # Entrenar modelo
    model.fit(X_train, y_train)
    # Hacer predicciones
    y_pred = model.predict(X_test)

    # Calcular métricas
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    # Guardar resultados
    results[name] = {
        'RMSE': rmse,
        'MAE': mae,
        'R2': r2
        }

In [ ]:
results_df = pd.DataFrame(results).T
print("\nResultados de los modelos:")
print(results_df)

In [ ]:
plt.figure(figsize=(12, 6))
results_df[['RMSE', 'MAE']].plot(kind='bar')
plt.title('Comparación de Métricas de Error por Modelo')
plt.ylabel('Error')
plt.xticks(rotation=360)
plt.tight_layout()
# guardar la imagen:
plt.savefig(RAIZ / "resultados" / "comparacion_metricas_error_modelos.png", dpi=200, bbox_inches='tight')
plt.show() # mostrarla

In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(results_df.index, results_df['R2'])
plt.title('Coeficiente de Determinación (R²) por Modelo')
plt.ylabel('R²')
plt.xticks(rotation=45)
plt.tight_layout()
# guardar la imagen:
plt.savefig(RAIZ / "resultados" / "comparacion_r2_modelos.png", dpi=200, bbox_inches='tight')
plt.show() # mostrarla

In [ ]:
best_model = models['Random Forest']

In [ ]:
feature_names = (numeric_features +
best_model.named_steps['preprocessor']
.named_transformers_['cat']
.get_feature_names_out(categorical_features).tolist())

importances = best_model.named_steps['regressor'].feature_importances_
importance_df = pd.DataFrame({
'feature': feature_names,
'importance': importances
})
importance_df = importance_df.sort_values('importance', ascending=False).head(10)

plt.figure(figsize=(12, 6))
sns.barplot(data=importance_df, x='importance', y='feature')
plt.title('Top 10 Características Más Importantes (Random Forest)')
plt.xlabel('Importancia')
plt.tight_layout()
# guardar la imagen:
plt.savefig(RAIZ / "resultados" / "importancia_caracteristicas_random_forest.png", dpi=200, bbox_inches='tight')
plt.show()  # mostrarla

In [ ]:
y_pred_rf = best_model.predict(X_test)
plt.figure(figsize=(10, 6))
plt.scatter(y_test, y_pred_rf, alpha=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel('Tráfico Promedio Diario Real')
plt.ylabel('Tráfico Promedio Diario Predicho')
plt.title('Predicciones vs Valores Reales (Random Forest)')
plt.tight_layout()
# guardar la imagen:
plt.savefig(RAIZ / "resultados" / "predicciones_vs_valores_reales_random_forest.png", dpi=200, bbox_inches='tight')
plt.show()  # mostrarla

#### Predecir en cascada `IngresoPromedioDiario`:

##### Modelado en cascada: predicción de IngresoPromedioDiario usando el tráfico predicho

Ahora implementaremos un pipeline de aprendizaje supervisado para predecir `IngresoPromedioDiario`, usando como variables explicativas:

- Las mismas variables originales (`ValorTarifa`, `TipoVehiculo`, `Peaje`, `DuracionPeriodo(DIAS)`)
- **Y** la columna de `TraficoPromedioDiario` predicha por el mejor modelo anterior (Random Forest), en vez del valor real.

Esto simula un escenario realista donde el tráfico promedio diario no es conocido, sino estimado por un modelo previo.

In [ ]:
df_filtered_2019_cats_oficiales.head(2)

In [ ]:
# 1. Crear columna de tráfico predicho en el set completo (usando el mejor modelo entrenado)
# Entrenamos el modelo Random Forest en TODO el dataset para obtener predicciones consistentes
from sklearn.base import clone

# Clonar el pipeline para evitar sobreescribir el modelo ya entrenado
global_rf_model = clone(models['Random Forest'])
global_rf_model.fit(X, y)

df_cascada = df_filtered_2019_cats_oficiales.copy()
df_cascada['TraficoPredicho_RF'] = global_rf_model.predict(X)

df_cascada[['TraficoPromedioDiario', 'TraficoPredicho_RF']].head()

In [ ]:
# 2. Definir variables para el nuevo modelo de IngresoPromedioDiario
features_cascada = ['ValorTarifa', 'TipoVehiculo', 'Peaje', 'DuracionPeriodo(DIAS)', 'TraficoPredicho_RF']
target_cascada = 'IngresoPromedioDiario'

X_cascada = df_cascada[features_cascada]
y_cascada = df_cascada[target_cascada]

# División train/test
X_train_cas, X_test_cas, y_train_cas, y_test_cas = train_test_split(X_cascada, y_cascada, test_size=0.2, random_state=42)

# Preprocesamiento: igual que antes
numeric_features_cas = ['ValorTarifa', 'TraficoPredicho_RF']
categorical_features_cas = ['TipoVehiculo', 'Peaje', 'DuracionPeriodo(DIAS)']

preprocessor_cas = ColumnTransformer([
    ('num', StandardScaler(), numeric_features_cas),
    ('cat', OneHotEncoder(drop='first', sparse_output=False), categorical_features_cas)
])

In [ ]:
# 3. Entrenar y evaluar modelos de regresión para IngresoPromedioDiario usando el tráfico predicho
models_cascada = {
    'Linear Regression': Pipeline([
        ('preprocessor', preprocessor_cas),
        ('regressor', LinearRegression())
    ]),
    'Random Forest': Pipeline([
        ('preprocessor', preprocessor_cas),
        ('regressor', RandomForestRegressor(n_estimators=100, random_state=42))
    ]),
    'XGBoost': Pipeline([
        ('preprocessor', preprocessor_cas),
        ('regressor', xgb.XGBRegressor(random_state=42))
    ]),
    'SVR': Pipeline([
        ('preprocessor', preprocessor_cas),
        ('regressor', SVR(kernel='rbf'))
    ])
}

results_cascada = {}

for name, model in models_cascada.items():
    model.fit(X_train_cas, y_train_cas)
    y_pred = model.predict(X_test_cas)
    mse = mean_squared_error(y_test_cas, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test_cas, y_pred)
    r2 = r2_score(y_test_cas, y_pred)
    results_cascada[name] = {'RMSE': rmse, 'MAE': mae, 'R2': r2}

results_cascada_df = pd.DataFrame(results_cascada).T
print("\nResultados modelos cascada (IngresoPromedioDiario):")
print(results_cascada_df)

In [ ]:
# 4. Visualización de resultados
plt.figure(figsize=(12, 6))
results_cascada_df[['RMSE', 'MAE']].plot(kind='bar')
plt.title('Comparación de Métricas de Error por Modelo (Cascada)')
plt.ylabel('Error')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print("\n\n")

plt.figure(figsize=(8, 5))
plt.bar(results_cascada_df.index, results_cascada_df['R2'])
plt.title('Coeficiente de Determinación (R²) por Modelo (Cascada)')
plt.ylabel('R²')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
best_model_cascada = models_cascada['XGBoost']  # el mejor esta vez, superando ligeramente a Random Forest...

In [ ]:
y_pred_xgb_cascada = best_model_cascada.predict(X_test_cas)
plt.figure(figsize=(10, 6))
plt.scatter(y_test_cas, y_pred_xgb_cascada, alpha=0.5)
plt.plot([y_test_cas.min(), y_test_cas.max()], [y_test_cas.min(), y_test_cas.max()], 'r--', lw=2)
plt.xlabel('Ingresos Promedios Diarios Reales')
plt.ylabel('Ingresos Promedios Diarios Predichos')
plt.title('Predicciones vs Valores Reales (XGBoost)')
plt.tight_layout()
plt.show()

### 3️⃣ Deep Learning: Red Neuronal Multicapa (MLP) para regresión

#### Predecir `TraficoPromedioDiario`:

##### 🅰️ ANN (MLP) con arquitectura simple:

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.utils import plot_model
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Preprocesamiento: OneHot para categóricas, escalado para numéricas
# Usar el mismo X, y definidos antes
X_num = X[['ValorTarifa']].values
X_cat = X[['TipoVehiculo', 'Peaje', 'DuracionPeriodo(DIAS)']].astype(str)

ohe = OneHotEncoder(sparse_output=False, drop='first')
X_cat_enc = ohe.fit_transform(X_cat)

scaler = StandardScaler()
X_num_scaled = scaler.fit_transform(X_num)

X_dl = np.hstack([X_num_scaled, X_cat_enc])

# División train/test igual que antes
X_train_dl, X_test_dl, y_train_dl, y_test_dl = train_test_split(X_dl, y, test_size=0.2, random_state=42)

# Definir modelo MLP simple
model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train_dl.shape[1],)),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(1)
])

model.compile(optimizer='adam', loss='mse', metrics=['mae'])

# Entrenar
history = model.fit(X_train_dl, y_train_dl, epochs=50, batch_size=32, validation_split=0.2, verbose=1)

# Evaluar
y_pred_dl = model.predict(X_test_dl).flatten()
rmse_dl = np.sqrt(mean_squared_error(y_test_dl, y_pred_dl))
mae_dl = mean_absolute_error(y_test_dl, y_pred_dl)
r2_dl = r2_score(y_test_dl, y_pred_dl)

print(f"MLP - RMSE: {rmse_dl:.2f} | MAE: {mae_dl:.2f} | R2: {r2_dl:.4f}")

In [ ]:
# Gráfico de la función de pérdida

plt.figure(figsize=(8,4))
plt.plot(history.history['loss'], label='train_loss')
plt.plot(history.history['val_loss'], label='val_loss')
plt.title('Evolución de la función de pérdida (MSE)')
plt.xlabel('Época')
plt.ylabel('MSE')
plt.legend()
plt.tight_layout()
# guardar la imagen:
plt.savefig(RAIZ / "resultados" / "evolucion_funcion_perdida_mlp.png", dpi=200, bbox_inches='tight')
plt.show() # mostrarla

##### 🅱️ Intentos por mejorar mi ANN:

###### Intento 1: ❌ fallido, pero rendimiento muy cercano al primer modelo

In [ ]:
from tensorflow.keras.layers import Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam


# --- Preprocesamiento idéntico ---
numeric_features = ['ValorTarifa']
categorical_features = ['TipoVehiculo', 'Peaje', 'DuracionPeriodo(DIAS)']

scaler = StandardScaler()
X_num_scaled = scaler.fit_transform(X[numeric_features])

ohe = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')
X_cat_encoded = ohe.fit_transform(X[categorical_features])

X_dl = np.hstack([X_num_scaled, X_cat_encoded])

X_train_dl, X_test_dl, y_train_dl, y_test_dl = train_test_split(
    X_dl, y, test_size=0.2, random_state=42
)

# --- Modelo EQUILIBRADO (versión mejorada pero sencilla) ---
model_balanced = Sequential([
    Dense(64, activation='relu', input_shape=(X_dl.shape[1],)),
    Dropout(0.2),

    Dense(32, activation='relu'),
    Dropout(0.2),

    Dense(1)
])

# Compilar con lr más bajo
initial_lr = 5e-4  # más suave que 1e-3
optimizer = Adam(learning_rate=initial_lr)
model_balanced.compile(optimizer=optimizer, loss='mse', metrics=['mae'])

# Callbacks más conservadores
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True,
    verbose=1
)

lr_scheduler = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    min_lr=1e-6,
    verbose=1
)

# Entrenamiento
print("Entrenando MLP equilibrado...")
history_balanced = model_balanced.fit(
    X_train_dl, y_train_dl,
    epochs=108,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop, lr_scheduler],
    verbose=1
)

# Evaluación
y_pred_balanced = model_balanced.predict(X_test_dl).flatten()

rmse_balanced = np.sqrt(mean_squared_error(y_test_dl, y_pred_balanced))
mae_balanced = mean_absolute_error(y_test_dl, y_pred_balanced)
r2_balanced = r2_score(y_test_dl, y_pred_balanced)

print(f"\nMLP EQUILIBRADO - RMSE: {rmse_balanced:.2f} | MAE: {mae_balanced:.2f} | R2: {r2_balanced:.4f}")

# Comparación con RF
best_model_rf = models['Random Forest']
y_pred_rf = best_model_rf.predict(X_test)
rmse_rf = np.sqrt(mean_squared_error(y_test_dl, y_pred_rf))
mae_rf = mean_absolute_error(y_test_dl, y_pred_rf)
r2_rf = r2_score(y_test_dl, y_pred_rf)

print(f"Random Forest (test) - RMSE: {rmse_rf:.2f} | MAE: {mae_rf:.2f} | R2: {r2_rf:.4f}")

# Tabla final
comparison_df = pd.DataFrame({
    'MLP Original': [727.60, 257.37, 0.8803],
    'MLP Equilibrado': [rmse_balanced, mae_balanced, r2_balanced],
    'Random Forest': [rmse_rf, mae_rf, r2_rf]
}, index=['RMSE', 'MAE', 'R²'])
print("\nComparación final:")
print(comparison_df.round(4))

# Gráfico de pérdida
plt.figure(figsize=(8,4))
plt.plot(history_balanced.history['loss'], label='train_loss')
plt.plot(history_balanced.history['val_loss'], label='val_loss')
plt.title('Evolución de la función de pérdida (MSE) - MLP Equilibrado')
plt.xlabel('Época')
plt.ylabel('MSE')
plt.legend()
plt.tight_layout()
# guardar la imagen:
plt.savefig(RAIZ / "resultados" / "evolucion_funcion_perdida_mlp_equilibrado.png", dpi=200, bbox_inches='tight')
plt.show() # mostrarla

###### Intento 2: ❌❌❌ fallido, bajón claro en el rendimiento respecto al primer modelo

In [ ]:
## Probar con un lr un poco más alto
from tensorflow.keras.optimizers import Adam

# Preprocesamiento: OneHot para categóricas, escalado para numéricas
# Usar el mismo X, y definidos antes
X_num = X[['ValorTarifa']].values
X_cat = X[['TipoVehiculo', 'Peaje', 'DuracionPeriodo(DIAS)']].astype(str)

ohe = OneHotEncoder(sparse_output=False, drop='first')
X_cat_enc = ohe.fit_transform(X_cat)

scaler = StandardScaler()
X_num_scaled = scaler.fit_transform(X_num)

X_dl = np.hstack([X_num_scaled, X_cat_enc])

# División train/test igual que antes
X_train_dl, X_test_dl, y_train_dl, y_test_dl = train_test_split(X_dl, y, test_size=0.2, random_state=42)

# Definir modelo MLP simple
model_original = Sequential([
    Dense(64, activation='relu', input_shape=(X_train_dl.shape[1],)),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(1)
])
optimizer = Adam(learning_rate=2e-3)  # 👈 ¡Subimos un poco el lr!
model_original.compile(optimizer=optimizer, loss='mse', metrics=['mae'])

# Entrenar
history_fast = model_original.fit(X_train_dl, y_train_dl, epochs=50, batch_size=32, validation_split=0.2, verbose=1)

# Evaluar
y_pred_fast = model_original.predict(X_test_dl).flatten()
rmse_fast = np.sqrt(mean_squared_error(y_test_dl, y_pred_fast))
mae_fast = mean_absolute_error(y_test_dl, y_pred_fast)
r2_fast = r2_score(y_test_dl, y_pred_fast)

print(f"MLP - RMSE: {rmse_fast:.2f} | MAE: {mae_fast:.2f} | R2: {r2_fast:.4f}")

# Gráfico de pérdida
plt.figure(figsize=(8,4))
plt.plot(history_fast.history['loss'], label='train_loss')
plt.plot(history_fast.history['val_loss'], label='val_loss')
plt.title('Evolución de la función de pérdida (MSE) - Lerning Rate más alto')
plt.xlabel('Época')
plt.ylabel('MSE')
plt.legend()
plt.tight_layout()
# guardar la imagen:
plt.savefig(RAIZ / "resultados" / "evolucion_funcion_perdida_mlp_lr_mas_alto.png", dpi=200, bbox_inches='tight')
plt.show()  # mostrarla

###### Intento 3: ✅✅✅ ÉXITO, mejora ligera del desempeño frente al primer modelo

In [ ]:
# Modelo original, pero con early stopping
model_simple_es = Sequential([
    Dense(64, activation='relu', input_shape=(X_dl.shape[1],)),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(1)
])

model_simple_es.compile(optimizer='adam', loss='mse', metrics=['mae'])

# Callbacks suaves
early_stop = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)
lr_reduce = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=8, min_lr=1e-6)

history_es = model_simple_es.fit(
    X_train_dl, y_train_dl,
    epochs=100,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop, lr_reduce],
    verbose=1
)

y_pred_es = model_simple_es.predict(X_test_dl).flatten()
rmse_es = np.sqrt(mean_squared_error(y_test_dl, y_pred_es))
mae_es = mean_absolute_error(y_test_dl, y_pred_es)
r2_es = r2_score(y_test_dl, y_pred_es)

print(f"MLP + ES - RMSE: {rmse_es:.2f} | MAE: {mae_es:.2f} | R2: {r2_es:.4f}")

# Gráfico de pérdida
plt.figure(figsize=(8,4))
plt.plot(history_es.history['loss'], label='train_loss')
plt.plot(history_es.history['val_loss'], label='val_loss')
plt.title('Evolución de la función de pérdida (MSE) - MLP + Early Stopping')
plt.xlabel('Época')
plt.ylabel('MSE')
plt.legend()
plt.tight_layout()
# guardar la imagen:
plt.savefig(RAIZ / "resultados" / "evolucion_funcion_perdida_mlp_early_stopping.png", dpi=200, bbox_inches='tight')
plt.show()  # mostrarla

#### Predecir en cascada `IngresoPromedioDiario`:

In [ ]:
# --- Modelado en cascada con Deep Learning (ANN) para predecir IngresoPromedioDiario ---

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# 1. Predecir TraficoPromedioDiario en TODO el dataset usando la ANN simple ya entrenada
# (Si el modelo se llama 'model', y el preprocesamiento es X_dl, ohe, scaler, etc.)

# Preprocesamiento para TODO el dataset
X_num_full = X[['ValorTarifa']].values
X_cat_full = X[['TipoVehiculo', 'Peaje', 'DuracionPeriodo(DIAS)']].astype(str)
X_cat_enc_full = ohe.transform(X_cat_full)
X_num_scaled_full = scaler.transform(X_num_full)
X_dl_full = np.hstack([X_num_scaled_full, X_cat_enc_full])

# Predicción de tráfico promedio diario (ANN)
trafico_predicho_ann = model.predict(X_dl_full).flatten()

# 2. Crear DataFrame extendido con la predicción
df_cascada_dl = df_filtered_2019_cats_oficiales.copy()
df_cascada_dl['TraficoPredicho_ANN'] = trafico_predicho_ann

# 3. Preparar variables para la ANN de IngresoPromedioDiario
features_cascada_dl = ['ValorTarifa', 'TipoVehiculo', 'Peaje', 'DuracionPeriodo(DIAS)', 'TraficoPredicho_ANN']
target_cascada_dl = 'IngresoPromedioDiario'

X_cascada_dl = df_cascada_dl[features_cascada_dl]
y_cascada_dl = df_cascada_dl[target_cascada_dl]

# Preprocesamiento: OneHot para categóricas, escalado para numéricas
X_num_cas = X_cascada_dl[['ValorTarifa', 'TraficoPredicho_ANN']].values
X_cat_cas = X_cascada_dl[['TipoVehiculo', 'Peaje', 'DuracionPeriodo(DIAS)']].astype(str)

ohe_cas = OneHotEncoder(sparse_output=False, drop='first')
X_cat_cas_enc = ohe_cas.fit_transform(X_cat_cas)

scaler_cas = StandardScaler()
X_num_cas_scaled = scaler_cas.fit_transform(X_num_cas)

X_dl_cas = np.hstack([X_num_cas_scaled, X_cat_cas_enc])

# División train/test
X_train_dl_cas, X_test_dl_cas, y_train_dl_cas, y_test_dl_cas = train_test_split(
    X_dl_cas, y_cascada_dl, test_size=0.2, random_state=42
)

# 4. Definir y entrenar la ANN para IngresoPromedioDiario
model_cascada = Sequential([
    Dense(64, activation='relu', input_shape=(X_train_dl_cas.shape[1],)),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(1)
])

model_cascada.compile(optimizer='adam', loss='mse', metrics=['mae'])

early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
lr_reduce = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6)

history_cascada = model_cascada.fit(
    X_train_dl_cas, y_train_dl_cas,
    epochs=60,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop, lr_reduce],
    verbose=1
)

# 5. Evaluar el modelo
y_pred_dl_cascada = model_cascada.predict(X_test_dl_cas).flatten()
rmse_dl_cascada = np.sqrt(mean_squared_error(y_test_dl_cas, y_pred_dl_cascada))
mae_dl_cascada = mean_absolute_error(y_test_dl_cas, y_pred_dl_cascada)
r2_dl_cascada = r2_score(y_test_dl_cas, y_pred_dl_cascada)

print(f"ANN Cascada - RMSE: {rmse_dl_cascada:.2f} | MAE: {mae_dl_cascada:.2f} | R2: {r2_dl_cascada:.4f}")

# 6. Gráfico de predicciones vs valores reales
plt.figure(figsize=(10, 6))
plt.scatter(y_test_dl_cas, y_pred_dl_cascada, alpha=0.5)
plt.plot([y_test_dl_cas.min(), y_test_dl_cas.max()], [y_test_dl_cas.min(), y_test_dl_cas.max()], 'r--', lw=2)
plt.xlabel('Ingresos Promedios Diarios Reales')
plt.ylabel('Ingresos Promedios Diarios Predichos (ANN Cascada)')
plt.title('Predicciones vs Valores Reales (ANN Cascada)')
plt.tight_layout()
plt.show()

# 7. Gráfico de la función de pérdida
plt.figure(figsize=(8,4))
plt.plot(history_cascada.history['loss'], label='train_loss')
plt.plot(history_cascada.history['val_loss'], label='val_loss')
plt.title('Evolución de la función de pérdida (MSE) - ANN Cascada')
plt.xlabel('Época')
plt.ylabel('MSE')
plt.legend()
plt.tight_layout()
# guardar la imagen:
plt.savefig(RAIZ / "resultados" / "evolucion_funcion_perdida_ann_cascada.png", dpi=200, bbox_inches='tight')
plt.show()  # mostrarla

In [ ]:
# --- ANN cascada mejorada para predecir IngresoPromedioDiario ---

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# Preprocesamiento idéntico al anterior (ya tienes X_dl_cas, y_cascada_dl, etc.)
# Si no existen, ejecuta la celda anterior primero.

# División train/test (por si necesitas redefinir)
# X_train_dl_cas, X_test_dl_cas, y_train_dl_cas, y_test_dl_cas = train_test_split(
#     X_dl_cas, y_cascada_dl, test_size=0.2, random_state=42)

# Arquitectura más profunda y regularizada
model_cascada_deep = Sequential([
    Dense(128, activation='relu', input_shape=(X_train_dl_cas.shape[1],)),
    Dropout(0.12),
    Dense(64, activation='relu'),
    Dropout(0.12),
    Dense(32, activation='relu'),
    Dropout(0.1),
    Dense(16, activation='relu'),
    Dense(1)
])

optimizer = Adam(learning_rate=1e-4)
model_cascada_deep.compile(optimizer=optimizer, loss='mse', metrics=['mae'])

# Callbacks
early_stop_deep = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)
lr_reduce_deep = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=7, min_lr=1e-6)

# Entrenamiento
history_cascada_deep = model_cascada_deep.fit(
    X_train_dl_cas, y_train_dl_cas,
    epochs=120,
    batch_size=64,
    validation_split=0.2,
    callbacks=[early_stop_deep, lr_reduce_deep],
    verbose=1
)

# Evaluación
y_pred_dl_cascada_deep = model_cascada_deep.predict(X_test_dl_cas).flatten()
rmse_dl_cascada_deep = np.sqrt(mean_squared_error(y_test_dl_cas, y_pred_dl_cascada_deep))
mae_dl_cascada_deep = mean_absolute_error(y_test_dl_cas, y_pred_dl_cascada_deep)
r2_dl_cascada_deep = r2_score(y_test_dl_cas, y_pred_dl_cascada_deep)

print(f"ANN Cascada Mejorada - RMSE: {rmse_dl_cascada_deep:.2f} | MAE: {mae_dl_cascada_deep:.2f} | R2: {r2_dl_cascada_deep:.4f}")

# Gráfico de predicciones vs valores reales
plt.figure(figsize=(10, 6))
plt.scatter(y_test_dl_cas, y_pred_dl_cascada_deep, alpha=0.5)
plt.plot([y_test_dl_cas.min(), y_test_dl_cas.max()], [y_test_dl_cas.min(), y_test_dl_cas.max()], 'r--', lw=2)
plt.xlabel('Ingresos Promedios Diarios Reales')
plt.ylabel('Ingresos Promedios Diarios Predichos (ANN Cascada Mejorada)')
plt.title('Predicciones vs Valores Reales (ANN Cascada Mejorada)')
plt.tight_layout()
# guardar la imagen:
plt.savefig(RAIZ / "resultados" / "predicciones_vs_valores_reales_ann_cascada_mejorada.png", dpi=200, bbox_inches='tight')
plt.show()  # mostrarla

# Gráfico de la función de pérdida
plt.figure(figsize=(8,4))
plt.plot(history_cascada_deep.history['loss'], label='train_loss')
plt.plot(history_cascada_deep.history['val_loss'], label='val_loss')
plt.title('Evolución de la función de pérdida (MSE) - ANN Cascada Mejorada')
plt.xlabel('Época')
plt.ylabel('MSE')
plt.legend()
plt.tight_layout()
# guardar la imagen:
plt.savefig(RAIZ / "resultados" / "evolucion_funcion_perdida_ann_cascada_mejorada.png", dpi=200, bbox_inches='tight')
plt.show()  # mostrarla

### 📊 Interpretación y comparación final de modelos en cascada para predecir IngresoPromedioDiario

**Resumen de resultados:**

- **XGBoost (cascada):** Mejor desempeño general para predecir ingresos diarios promedio, con mayor R² y menor error (RMSE, MAE).
- **Random Forest (cascada):** Muy competitivo, resultados similares a XGBoost.
- **ANN cascada simple:** Desempeño aceptable, pero menor capacidad predictiva (R² más bajo, errores más altos).
- **ANN cascada mejorada:** Mejora respecto a la ANN simple, pero sigue por debajo de los modelos de árbol.

**¿Por qué los modelos de árbol superan a las ANN en este caso?**
- Los datos tabulares con muchas variables categóricas y relaciones no lineales suelen favorecer a modelos como XGBoost y Random Forest.
- Las ANN requieren más datos, ajuste fino y pueden ser sensibles a la propagación de errores en cascada.
- Los modelos de árbol manejan mejor la heterogeneidad y la interacción entre variables categóricas.

**Recomendación para el póster:**
- Resalta que, aunque las redes neuronales mejoran con mayor profundidad y regularización, los modelos de árbol siguen siendo la mejor opción para este tipo de predicción en cascada sobre datos tabulares.
- Incluye las métricas clave (RMSE, MAE, R²) de cada modelo en una tabla comparativa.
- Menciona que la cascada ANN puede ser útil en otros contextos, pero aquí XGBoost y RF son más robustos y precisos.

**Conclusión:**
> Para predecir ingresos diarios promedio en peajes colombianos usando un pipeline en cascada, los modelos de árbol (XGBoost, Random Forest) superan consistentemente a las redes neuronales multicapa, tanto en precisión como en estabilidad.